# Evaluating the Segmentation Models

In [12]:
# ======================
# Load Libraries
# ======================

import os
import re
import torch
import numpy as np
import pandas as pd
from skimage.io import imread
from skimage.metrics import hausdorff_distance
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
import torchvision.transforms.functional as TF

import sys
sys.path.append('../')
from utils.models.uNet import UNet
from utils.models.SegFormer import segformer
from utils.models.deeplabv3p import DeepLabV3Plus
from utils.models.SegNet import segnet
from utils.models.maskFormer import MaskFormer
from utils.models.resnet101 import resnet101_backbone

In [2]:
# ======================
# Utility functions
# ======================

def dice_coefficient(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection) / (np.sum(y_true) + np.sum(y_pred) + 1e-7)

def iou_score(y_true, y_pred):
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return intersection / (union + 1e-7)

def pixel_accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / y_true.size

def mean_absolute_error(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def compute_metrics(y_true, y_pred):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    precision = precision_score(y_true_flat, y_pred_flat, zero_division=0)
    recall = recall_score(y_true_flat, y_pred_flat, zero_division=0)
    f1 = f1_score(y_true_flat, y_pred_flat, zero_division=0)
    iou = iou_score(y_true, y_pred)
    dice = dice_coefficient(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    hd = hausdorff_distance(y_true, y_pred)
    acc = pixel_accuracy(y_true, y_pred)
    
    return iou, dice, precision, recall, f1, mae, hd, acc

In [5]:
# ======================
# Model loading stubs
# ======================

def load_model(crop, model_name, device):
    if model_name == "U-NET":
        channels = [32, 64, 128, 256, 512]
        model = UNet(in_channels=3, out_channels=1, channels=channels, bilinear=True, use_batchnorm=True).to(device)
    elif model_name == "SegFormer":
        model = segformer(in_channels = 3, num_classes = 1).to(device)
    elif model_name == "DeepLabV3Plus":
        model = DeepLabV3Plus(num_classes=1, output_stride=16, backbone_width_mult=1.0).to(device)
    elif model_name == "SegNet":
        model = segnet(in_channels=3, num_classes=1, pretrained=False).to(device)
    elif model_name == "MaskFormer":
        resnet101 = resnet101_backbone()
        model = MaskFormer(
            backbone=resnet101, 
                num_classes=1, 
                num_queries=5,
                embed_dim=64, 
                transformer_layers=1,
                transformer_heads=2,
                transformer_ffn_dim=256,
                return_binary=True
            ).to(device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    model.load_state_dict(torch.load(f'../models/{crop}_{model_name}_seg.pt', map_location=device))
    model.eval()
    return model

def predict_mask(model, image, device):
    img_input = image.unsqueeze(0).to(device)
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            pred_logits = model(img_input)
            pred_mask = (torch.sigmoid(pred_logits) > 0.5).float().cpu().squeeze().numpy()
    return pred_mask

In [6]:
from torch.utils.data import Dataset, DataLoader

class SegEvalDataset(Dataset):
    def __init__(self, image_dir, mask_dir, image_files):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_files = image_files

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        mask_name = os.path.splitext(img_name)[0] + ".png"
        image = imread(os.path.join(self.image_dir, img_name))
        mask = imread(os.path.join(self.mask_dir, mask_name))
        if mask.ndim == 3:
            mask = mask[..., 0]
        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask).float().unsqueeze(0)
        return image, mask, img_name

## Using initial trained models

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "../data/"
crops = ["sorghum", "wheat"]
models = ["DeepLabV3Plus", "SegFormer", "SegNet", "U-NET"]

In [9]:
# ======================
# Main evaluation
# ======================

results = []

BATCH_SIZE = 8

for crop in crops:
    image_dir = os.path.join(data_dir, crop, "test", "images")
    mask_dir = os.path.join(data_dir, crop, "test", "masks")
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    dataset = SegEvalDataset(image_dir, mask_dir, image_files)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    for model_name in models:
        print(f"Evaluating {model_name} on {crop}...")
        model = load_model(crop, model_name, device)
        model.eval()

        for images, masks, img_names in tqdm(loader):
            images = images.to(device)
            masks_np = masks.squeeze(1).cpu().numpy()  # [B, H, W]
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy().squeeze(1)  # [B, H, W]

            for i in range(images.size(0)):
                mask = masks_np[i]
                pred = preds[i]
                img_name = img_names[i]
                match = re.match(r'(\d+)_([^_]+)_\d+(_aug)?', img_name)
                if match:
                    collectiondate, genotype, aug = match.groups()
                    augmented = "yes" if aug else "no"
                else:
                    collectiondate, genotype, augmented = "unknown", "unknown", "unknown"
                iou, dice, precision, recall, f1, mae, hd, acc = compute_metrics(mask, pred)
                results.append({
                    "crop": crop,
                    "collectiondate": collectiondate,
                    "genotype": genotype,
                    "augmented?": augmented,
                    "modelname": model_name,
                    "IoU": iou,
                    "Dice": dice,
                    "Precision": precision,
                    "Recall": recall,
                    "F1": f1,
                    "MAE": mae,
                    "Hausdorff": hd,
                    "PixelAccuracy": acc
                })

Evaluating DeepLabV3Plus on sorghum...


100%|██████████| 861/861 [20:58<00:00,  1.46s/it]


Evaluating SegFormer on sorghum...


100%|██████████| 861/861 [20:48<00:00,  1.45s/it]


Evaluating SegNet on sorghum...


...

100%|██████████| 444/444 [23:24<00:00,  3.16s/it]   


Evaluating SegNet on wheat...


100%|██████████| 444/444 [16:57<00:00,  2.29s/it]


Evaluating U-NET on wheat...


100%|██████████| 444/444 [11:34<00:00,  1.56s/it]


In [10]:
# ======================
# Save to CSV
# ======================

df = pd.DataFrame(results)
df.to_csv("../results/model_evaluation_results.csv", index=False)
print("✅ Evaluation completed and saved to results/model_evaluation_results.csv")

✅ Evaluation completed and saved to results/model_evaluation_results.csv


## Appending new trained models

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "../data/"
crops = ["sorghum", "wheat"]
models = ["MaskFormer"]

In [13]:
# ======================
# Main evaluation
# ======================

results = []

BATCH_SIZE = 8

for crop in crops:
    image_dir = os.path.join(data_dir, crop, "test", "images")
    mask_dir = os.path.join(data_dir, crop, "test", "masks")
    image_files = [f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]

    dataset = SegEvalDataset(image_dir, mask_dir, image_files)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    for model_name in models:
        print(f"Evaluating {model_name} on {crop}...")
        model = load_model(crop, model_name, device)
        model.eval()

        for images, masks, img_names in tqdm(loader):
            images = images.to(device)
            masks_np = masks.squeeze(1).cpu().numpy()  # [B, H, W]
            with torch.no_grad():
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    preds = (torch.sigmoid(logits) > 0.5).float().cpu().numpy().squeeze(1)  # [B, H, W]

            for i in range(images.size(0)):
                mask = masks_np[i]
                pred = preds[i]
                img_name = img_names[i]
                match = re.match(r'(\d+)_([^_]+)_\d+(_aug)?', img_name)
                if match:
                    collectiondate, genotype, aug = match.groups()
                    augmented = "yes" if aug else "no"
                else:
                    collectiondate, genotype, augmented = "unknown", "unknown", "unknown"
                iou, dice, precision, recall, f1, mae, hd, acc = compute_metrics(mask, pred)
                results.append({
                    "crop": crop,
                    "collectiondate": collectiondate,
                    "genotype": genotype,
                    "augmented?": augmented,
                    "modelname": model_name,
                    "IoU": iou,
                    "Dice": dice,
                    "Precision": precision,
                    "Recall": recall,
                    "F1": f1,
                    "MAE": mae,
                    "Hausdorff": hd,
                    "PixelAccuracy": acc
                })

Evaluating MaskFormer on sorghum...


c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
100%|██████████| 861/861 [23:42<00:00,  1.65s/it]


Evaluating MaskFormer on wheat...


c:\Users\gnoceras\Documents\SegmentationStudy\.venv\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
100%|██████████| 444/444 [07:51<00:00,  1.06s/it]


In [14]:
# ======================
# Save to CSV
# ======================

csv_path = "../results/model_evaluation_results_allData.csv"
try:
    df_old = pd.read_csv(csv_path)
except FileNotFoundError:
    df_old = pd.DataFrame()

# Create new results DataFrame
df_new = pd.DataFrame(results)

# Concatenate old and new results
df_all = pd.concat([df_old, df_new], ignore_index=True)

# Save combined DataFrame
df_all.to_csv(csv_path, index=False)
print("✅ Evaluation results appended and saved to results/model_evaluation_results_allData.csv")

✅ Evaluation results appended and saved to results/model_evaluation_results_allData.csv
